# 13 · Selección de características mediante Clustered MDA

El modelo del 11 utiliza diez características. Ahora estudiaremos si seleccionar grupos que aportan información mejora sus probabilidades. Las variables se agrupan porque varias pueden describir algo parecido: al alterar una sola, las demás podrían sustituirla y ocultar su importancia.

**Entradas:** tablas de 01 → 02 → 07 → 08 → 10 → 11, con el mismo entorno `.venv` y el extra `models`. El 09 y el 12 no son entradas del cálculo. SciPy ya está instalada con el modelado, no necesitas instalar otra biblioteca.

Solo se utiliza **1927–1989**. Los resultados de las sombras del 12 pertenecen al modelo de diez características, no se trasladan a esta variante con selección. El [protocolo](../docs/seleccion_clustered_mda.md) fija las decisiones antes de ejecutarla.


## 1. Una separación dentro del entrenamiento

Cada partición exterior del 10 reserva dos bloques para evaluar y utiliza los otros cuatro para entrenar, después de purga y embargo. Dentro de esos cuatro, ahora reservamos el último bloque para seleccionar los grupos. Los tres anteriores sirven para agrupar las variables y ajustar la logística interna.

La selección solo puede ver este entrenamiento y validación internos. Después reajustaremos con las columnas elegidas sobre todo el entrenamiento exterior original. Los dos bloques exteriores reservados se usarán para predecir y evaluar. Así, el examen final de esa partición no decide qué variables conservar.

Se mantienen las exclusiones exteriores y se aplica la separación interna sobre meses calendario. Usamos una única división interna fija por partición, su dependencia del periodo elegido es una limitación que examinaremos entre las quince particiones.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display

from tfm.data import project_root, save_result
from tfm.logistic import load_logistic_inputs
from tfm.clustered_mda import (CORRELATION_THRESHOLD, N_REPEATS, SEED,
                               build_inner_splits, cluster_features, run_clustered_mda)

ROOT = project_root()
ejemplos, particiones, _, _ = load_logistic_inputs(ROOT)
particiones_internas = build_inner_splits(particiones)
PARTICION_EJEMPLO = 1
REGLA_EJEMPLO = "momentum"
calendario = particiones_internas.loc[particiones_internas.split_id.eq(PARTICION_EJEMPLO)].copy()
display(pd.Series({"Partición exterior": PARTICION_EJEMPLO,
                   "Bloque reservado para selección interna": int(calendario.inner_validation_block.iloc[0]),
                   "Umbral de correlación absoluta": CORRELATION_THRESHOLD,
                   "Permutaciones por grupo": N_REPEATS, "Semilla base": SEED}, name="Diseño fijado"))
display(calendario.groupby(["role", "inner_role"]).size().rename("Meses de calendario").reset_index())

colores = {"outer_validation": ("Evaluación exterior", "#244b70"),
           "outer_purge": ("Purga exterior", "#b74e40"),
           "outer_embargo": ("Embargo exterior", "#d99b42"),
           "inner_train": ("Entrenamiento interno", "#9dc8b4"),
           "inner_validation": ("Selección interna", "#8770b2"),
           "inner_purge": ("Purga interna", "#d68d82"),
           "inner_embargo": ("Embargo interno", "#eacb91")}
papel = calendario.inner_role.where(calendario.role.eq("train"), "outer_" + calendario.role)
fig, ax = plt.subplots(figsize=(12, 3.2))
fechas = pd.PeriodIndex(calendario.target_period, freq="M").to_timestamp()
ax.scatter(fechas, np.zeros(len(calendario)), c=[colores[p][1] for p in papel], marker="|", s=230)
ax.set(title="Separación interna dentro de una partición exterior", xlabel="Mes objetivo", yticks=[], ylim=(-.5, .5))
fig.legend(handles=[Patch(color=color, label=nombre) for nombre, color in colores.values()],
           loc="lower center", bbox_to_anchor=(.5, .02), ncol=4, fontsize=8)
fig.subplots_adjust(bottom=.42, top=.85)
plt.show()


## 2. Agrupar sin consultar las etiquetas de validación

Calculamos correlaciones de Spearman únicamente sobre los ejemplos internos de entrenamiento de cada regla. Spearman compara sus posiciones relativas y recoge relaciones monótonas. Usamos el valor absoluto para agrupar también variables que evolucionan en sentidos opuestos.

La agrupación jerárquica de enlace completo utiliza la mayor disimilitud entre miembros. Fijamos una correlación absoluta de 0,70, sin probar después otros cortes para mejorar las métricas. Mantendremos las variables originales de los grupos seleccionados. Las columnas constantes se identifican aparte y no se les inventa una correlación.

La siguiente figura describe solo la partición y regla elegidas para inspección. La agrupación se repetirá dentro de cada entrenamiento interno, no se fijará utilizando todo el desarrollo. Sus identificadores podrán representar composiciones distintas entre particiones.


In [ ]:
filas = ejemplos.loc[ejemplos.strategy.eq(REGLA_EJEMPLO) & ejemplos.sample_usable]
filas = filas.merge(calendario[["target_period", "inner_role"]], on="target_period", validate="one_to_one")
entrenamiento_interno = filas.loc[filas.inner_role.eq("inner_train")]
grupos_ejemplo, constantes, correlacion = cluster_features(entrenamiento_interno)
display(pd.DataFrame({"Grupo": range(1, len(grupos_ejemplo) + 1),
                      "Características": [", ".join(g) for g in grupos_ejemplo]}))
print("Columnas constantes en el entrenamiento interno:", constantes)
if len(correlacion):
    fig, ax = plt.subplots(figsize=(8, 6))
    imagen = ax.imshow(correlacion, vmin=-1, vmax=1, cmap="RdBu_r")
    nombres = [c.removeprefix("z_") for c in correlacion.columns]
    ax.set_xticks(range(len(nombres)), nombres, rotation=90, fontsize=8)
    ax.set_yticks(range(len(nombres)), nombres, fontsize=8)
    ax.set_title("Spearman en entrenamiento interno: " + REGLA_EJEMPLO)
    fig.colorbar(imagen, ax=ax, label="Correlación de rangos")
    fig.tight_layout()
    plt.show()


## 3. Qué mide Clustered MDA

Entrenamos la logística con las características elegibles internas y calculamos su log-loss en el bloque interno reservado. Para cada grupo alteramos juntas sus columnas, utilizando el mismo orden de filas para todos sus miembros. El resto de características y las etiquetas no cambian. **El modelo permanece ajustado: no lo reentrenamos después de cada permutación.**

La importancia es la log-loss después de permutar menos la log-loss original. Un resultado positivo indica que alterar el grupo perjudica al modelo. Repetimos treinta veces y conservamos el grupo completo si la media es estrictamente positiva. La dispersión entre permutaciones es descriptiva, no un intervalo de confianza ni otro filtro estadístico.

Si ningún grupo aporta una media positiva, el modelo exterior tendrá solo intercepto: predecirá la frecuencia de éxito de su entrenamiento. Una selección no disponible por falta de datos es un caso distinto y quedará marcada, sin mover las fronteras.

Esta permutación mide dependencia del modelo respecto a un grupo. No conserva la autocorrelación y no es el contraste temporal con sombras circulares del 12. Una característica poco importante para esta logística puede ser útil para otro modelo.


In [ ]:
def mostrar_progreso(completadas, total):
    print(f"Particiones exteriores evaluadas: {completadas}/{total}", flush=True)

tablas, informe = run_clustered_mda(ROOT, progress=mostrar_progreso)
grupos = tablas["mda_feature_groups"]
predicciones = tablas["mda_logistic_predictions"]
mensual = tablas["mda_logistic_monthly"]
metricas = tablas["mda_logistic_metrics"]
coeficientes = tablas["mda_logistic_coefficients"]
diagnosticos = pd.DataFrame(informe["diagnostics"])
display(diagnosticos[["split_id", "strategy", "inner_validation_block", "inner_train_examples",
                     "inner_validation_examples", "inner_log_loss", "inner_naive_log_loss",
                     "selected_columns", "selection_status", "outer_model_status"]])
print("Estado del recorrido:", informe["experiment_status"])


## 4. Leer las importancias por grupo

Cada grupo tiene una importancia conjunta, no es la importancia individual de cada miembro. La tabla plana repite el estadístico en sus características para facilitar la consulta. Para el gráfico mostramos cada grupo una sola vez.

Las barras representan el incremento medio de log-loss y las líneas su desviación típica entre permutaciones. Estar a la derecha de cero indica que el modelo depende del grupo en esta validación interna. La calidad final se comprobará en los bloques exteriores, que no han decidido la selección.


In [ ]:
inspeccion = grupos.loc[grupos.split_id.eq(PARTICION_EJEMPLO) & grupos.strategy.eq(REGLA_EJEMPLO)]
display(inspeccion[["feature", "cluster_id", "group_members", "mean_delta_log_loss",
                    "std_delta_log_loss", "positive_repeats", "permutations", "selected", "status"]])
medidos = inspeccion.loc[inspeccion.status.eq("measured")].drop_duplicates("cluster_id")
if len(medidos):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh([f"Grupo {int(i)}" for i in medidos.cluster_id], medidos.mean_delta_log_loss,
            xerr=medidos.std_delta_log_loss,
            color=["#43876b" if v else "#b65b49" for v in medidos.selected])
    ax.axvline(0, color="black", linewidth=.8)
    ax.set(title="Importancia conjunta: media y desviación entre permutaciones",
           xlabel="Incremento de log-loss al permutar", ylabel="Grupo de esta partición y regla")
    fig.tight_layout()
    plt.show()


## 5. Cuánto cambia la selección entre particiones

Los grupos se aprenden en entrenamientos distintos y pueden cambiar de composición. No comparamos grupos solo porque tengan el mismo número. Mostramos cuántas de las quince particiones conservan cada característica, como descripción de estabilidad.

Esta frecuencia no es una probabilidad de que una variable sea importante. Tampoco la utilizamos para construir una lista global y volver a evaluar sobre los mismos bloques. Si falta alguna selección, su frecuencia completa queda no disponible. Una variable no seleccionada no equivale a un coeficiente estimado de cero.


In [ ]:
frecuencia = (grupos.groupby(["strategy", "feature"], sort=True)
              .agg(conservan=("selected", "sum"), disponibles=("selected", "count"),
                   previstas=("split_id", "nunique")).reset_index())
frecuencia["frecuencia"] = np.where(frecuencia.disponibles.eq(frecuencia.previstas),
                                    frecuencia.conservan / frecuencia.previstas, np.nan)
display(frecuencia)
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
for ax, regla in zip(axes, ("momentum", "reversal")):
    datos = frecuencia.loc[frecuencia.strategy.eq(regla)].sort_values("feature")
    ax.barh([c.removeprefix("z_") for c in datos.feature], datos.frecuencia.to_numpy(dtype=float) * 100)
    ax.set(title=regla, xlabel="Particiones que conservan la característica (%)", xlim=(0, 100))
    ax.grid(axis="x", alpha=.25)
fig.tight_layout()
plt.show()


## 6. Comparación exterior con las diez características

Con los grupos elegidos, se reajusta utilizando todo el entrenamiento exterior original, incluidas las observaciones que solo se habían separado para la selección interna. Se respetan las exclusiones exteriores. Las cinco predicciones de cada ejemplo se resumen igual que en el 11.

La tabla compara la logística con selección con el modelo de diez características y con el ingenuo. Una log-loss menor es favorable, una AUC mayor indica mejor ordenación de etiquetas. Conservar menos variables no garantiza una mejora. La comparación sigue siendo de descubrimiento y no simula una cartera continua.


In [ ]:
principal = metricas.loc[metricas.aggregation.eq("monthly_mean")]
original = pd.DataFrame(informe["original_metrics"])
comparacion = []
for regla in ("momentum", "reversal"):
    nuevo = principal.loc[principal.strategy.eq(regla) & principal.model.eq("logistic")].iloc[0]
    anterior = original.loc[original.strategy.eq(regla) & original.model.eq("logistic")].iloc[0]
    ingenuo = principal.loc[principal.strategy.eq(regla) & principal.model.eq("naive")].iloc[0]
    comparacion.append({"Regla": regla, "Ejemplos": nuevo.examples,
        "Log-loss con selección": nuevo.log_loss, "Log-loss con diez": anterior.log_loss,
        "Log-loss ingenuo": ingenuo.log_loss, "Mejora frente a diez (positiva es favorable)": anterior.log_loss - nuevo.log_loss,
        "AUC con selección": nuevo.auc, "AUC con diez": anterior.auc, "Estado": nuevo.status})
display(pd.DataFrame(comparacion))
display(metricas.loc[metricas.aggregation.eq("partition"),
                    ["split_id", "strategy", "model", "examples", "log_loss", "auc", "status"]])


## 7. Qué queda pendiente

La selección se limita a este candidato de logística, las diez características seguirán siendo candidatas para random forest o boosting. El criterio con sombras del 12 no valida automáticamente esta nueva variante: un contraste posterior deberá repetir también la selección dentro de las sombras.

Los resultados no fijan todavía una lista global definitiva ni acreditan rentabilidad. Continúan pendientes los otros modelos, SHAP y la evaluación económica con decisiones de aceptación, además de DSR/PBO y la evaluación desde 1990. Si esta selección empeora el modelo, se documentará sin retocar el umbral, la semilla o el bloque interno para obtener otra conclusión.

La celda siguiente guarda cinco tablas con sus JSON. El historial de esta variante queda en `outputs/experiments/clustered_mda_trials.jsonl`, las particiones y las permutaciones no cuentan como configuraciones independientes. Se conservan las salidas e historiales del 11 y 12. Si falla la celda de guardado, repítela completa antes de utilizar las salidas. Antes del commit, limpia las salidas y guarda.

Referencias: guía del tutor, López de Prado (2018), sección 8.3.2, y *Clustered Feature Importance* (2020). El agrupamiento concreto, la división interna y la selección por importancia media positiva son decisiones documentadas en [el protocolo](../docs/seleccion_clustered_mda.md). No se copia código externo.


In [ ]:
guardados = []
for nombre, tabla in tablas.items():
    salida = save_result(ROOT, nombre, tabla, informe)
    guardados.append({"Tabla": nombre + ".parquet", "Filas": salida["rows"]})
display(pd.DataFrame(guardados))
print("Historial:", informe["trial_log"])
print("Sombras de la nueva variante:", informe["skill_gate_status"])
